# infra-defect-detection — Phase 2 on Kaggle (YOLO11 single-country baseline, Czech)

**重要:跑这个notebook之前,先在右侧 Session options / Settings 里把 Accelerator 切成 GPU（T4 x2 或 P100）**——默认是CPU-only,YOLO训练会慢得不实际。

对应路线图阶段2:在单一国家(选Czech,1072张图,phase 1 EDA里确认分辨率统一600x600、数据量最小、能最快跑通train/eval/confusion matrix全链路)上训练YOLO11 baseline,建立"同国家内训练/评测"的基准分数——这是phase 3跨国分布漂移实验要对比的基线,不是最终目标。

产出要求(不是只报一个mAP):
- mAP@50 / mAP@50-95
- 混淆矩阵
- 按类别的precision/recall/AP
- 训练脚本+超参数+随机种子+数据划分全部记录,保证可复现

## 目录
0. 写入脚本(前两个和phase 1一致;extract_convert_per_country.py新增--countries筛选;新增make_country_split.py和train_cnn_baseline.py)
1. 安装依赖(ultralytics + phase 1的依赖)
2. 下载 + MD5校验(FigShare只有一个合并包,没法只下载Czech,但只解压转换Czech,很快)
3. 按国家分批解压转换,只跑Czech
4. 划分train/val/test(固定随机种子,记录划分结果)
5. 训练 + 在held-out test集上评测(YOLO11n,mAP+混淆矩阵+按类别P/R)
6. 核对结果
7. 收尾:提交并把小文件(report、metrics.json、图表)带回Mac


## 0. 写入脚本

In [ ]:
import os
os.makedirs("scripts", exist_ok=True)
os.makedirs("data", exist_ok=True)


In [ ]:
%%writefile scripts/download_rdd2022.py
"""Download and extract the RDD2022 road damage dataset (roadmap: infra-defect-detection, phase 1).

RDD2022 covers six countries (Japan, India, Czech Republic, Norway, United States, China) with
47,420 road images and 55,000+ annotated damage instances across four classes (D00 longitudinal
crack, D10 transverse crack, D20 alligator crack, D40 pothole). Images are CC BY-SA 4.0 per the
sekilab/RoadDamageDetector GitHub README (attribute sekilab/RoadDamageDetector + the RDD2022 paper,
arxiv.org/abs/2209.08538, in any README/Model Card that uses this data) - note the FigShare listing
below shows "CC BY 4.0" for the same dataset; this discrepancy between the two official sources is
unresolved, so treat CC BY-SA 4.0 (the more restrictive of the two, and the one stated by the
dataset's own authors on their own repo) as the operative license until/unless clarified.

SOURCE CHANGE (2026-09-16): this originally downloaded seven per-country zips from Sekilab's own S3
bucket (bigdatacup.s3.ap-northeast-1.amazonaws.com/.../Country_Specific_Data_CRDDC2022/...). That
bucket now returns HTTP 403 Forbidden on every file, confirmed independently from two unrelated
networks - the bucket's access policy appears to have changed, not a transient fluke. This version
instead downloads FigShare's official combined mirror of the same dataset (one zip, all six
countries, published by the RDD2022/CRDDC2022 organizers themselves), verified by MD5 checksum
against FigShare's own published hash so a partial/corrupted 12GB+ download is caught rather than
silently producing bad data. If FigShare's link ever breaks too, check
https://github.com/sekilab/RoadDamageDetector for current mirror links before assuming this script
is broken.

Because this is one combined zip (not one zip per country), there's no way to download only a
subset of countries - the whole ~12.35GB has to come down regardless. --countries now only controls
which countries get linked into data/raw/ for the conversion step afterward (convert_voc_to_yolo.py
reads data/raw/<country>/), not what gets downloaded.

The zip's exact internal folder layout was not independently verified before writing this script
(Sekilab's own "Directory_Structure_CRDDC_RDD2022.txt" reference file was unreachable from this
environment - see reorganize_countries() below for how this script copes with that uncertainty at
extraction time instead of assuming a fixed layout).

NOTE (2026-09-17, Kaggle handoff): on the original Mac/home-network run, aria2c's multi-connection
mode was confirmed to get an immediate HTTP 403 from FigShare's CDN regardless of connection count
(1, 4, or 16) - the CDN appears to reject Range/segmented requests outright, not just high
concurrency. So on Kaggle this will (correctly) fall back to the plain single-connection urllib
path every time; that's expected, not a bug - the win here is Kaggle's raw single-connection
bandwidth to this host, not multi-connection parallelism.

Usage:
    python scripts/download_rdd2022.py                        # download + extract + link all found countries
    python scripts/download_rdd2022.py --countries Japan Czech # only link these two after extraction
    python scripts/download_rdd2022.py --skip-extract          # download (+ verify) only
    python scripts/download_rdd2022.py --connections 32        # more aria2c connections (default 16)
"""
import argparse
import hashlib
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_ZIP_NAME = "RDD2022_released_through_CRDDC2022.zip"
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Logical country name -> name(s) we'll look for (case-insensitively) among directories inside the
# extracted zip, since the exact internal layout wasn't independently verified (see module docstring).
COUNTRY_ALIASES = {
    "Japan": ["Japan"],
    "India": ["India"],
    "Czech": ["Czech"],
    "Norway": ["Norway"],
    "United_States": ["United_States", "United States", "US", "USA"],
    "China_MotorBike": ["China_MotorBike", "China-MotorBike", "China_Motorbike"],
    "China_Drone": ["China_Drone", "China-Drone"],
}

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


def download_with_aria2(url, dest_path, connections=16, retries=3):
    """Segmented, multi-connection download via the aria2c CLI - typically several times faster than
    a single urllib stream on hosts like FigShare's S3-backed CDN, where per-connection throughput
    is often capped well below the link's actual bandwidth. aria2c handles its own retries/resume
    (via its .aria2 control file next to the output), so this just shells out and lets it manage
    that; --continue=true means re-running after an interrupted aria2c download resumes rather than
    restarting from zero.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "aria2c",
        "-x", str(connections),          # max connections per server
        "-s", str(connections),          # split file into this many pieces
        "-k", "1M",                      # min split size
        "--continue=true",
        "--max-tries", str(retries),
        "--retry-wait=5",
        "--file-allocation=none",
        "--summary-interval=5",
        # FigShare's CDN (via ndownloader.figshare.com -> a presigned S3 URL) returns 403 to aria2c's
        # default "aria2/x.y.z" User-Agent - matching the header the plain-urllib path already used
        # successfully fixes it. --auto-file-renaming=false avoids aria2 silently writing to a
        # "-1" suffixed file if dest_path.name already exists from an earlier failed attempt.
        "--user-agent=Mozilla/5.0",
        "--auto-file-renaming=false",
        "-d", str(dest_path.parent),
        "-o", dest_path.name,
        url,
    ]
    print(f"  using aria2c with {connections} parallel connections (much faster than a single stream)")
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"aria2c exited with code {result.returncode} - see its output above for details")


def download_one(url, dest_path, retries=3, connections=16):
    """Download url to dest_path. Skips entirely if dest_path already exists and is non-empty - this
    does NOT check the existing file's checksum, so a corrupted prior download won't be caught here;
    verify_md5() below is what actually guarantees integrity, run separately after this.

    Uses aria2c (multi-connection, much faster) when it's installed on PATH; otherwise falls back to
    a plain single-connection urllib download and prints a one-time hint about installing aria2c for
    a large file like this one. `brew install aria2` on macOS. If aria2c is present but fails (a
    misbehaving CDN, a network that blocks it, etc.), falls back to the urllib path automatically
    rather than giving up outright - not verified against every possible aria2c/network combination,
    so this fallback matters.
    """
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"  already have {dest_path.name} ({_format_bytes(dest_path.stat().st_size)}), skipping download")
        return

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("aria2c"):
        try:
            download_with_aria2(url, dest_path, connections=connections, retries=retries)
            return
        except RuntimeError as exc:
            print(f"  aria2c failed ({exc}); falling back to a plain single-connection download instead.")
            # clean up whatever partial/zero-byte file aria2c may have left behind before falling back
            if dest_path.exists() and dest_path.stat().st_size == 0:
                dest_path.unlink()
    else:
        print("  NOTE: aria2c not found on PATH - using a single-connection download, which will be "
              "noticeably slower for a file this size. `brew install aria2` (macOS) and re-run for a "
              "multi-connection download instead.")

    _download_urllib(url, dest_path, retries=retries)


def _download_urllib(url, dest_path, retries=3):
    """Plain single-connection streaming download - the fallback used when aria2c isn't available or
    didn't work."""
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")

    for attempt in range(1, retries + 1):
        try:
            print(f"  downloading {url} -> {dest_path} (attempt {attempt}/{retries})")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_path, "wb") as out:
                total = int(resp.headers.get("Content-Length", 0))
                downloaded = 0
                chunk_size = 1024 * 1024
                last_print = time.time()
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    out.write(chunk)
                    downloaded += len(chunk)
                    if time.time() - last_print > 2:
                        pct = f"{downloaded / total:.0%}" if total else "?"
                        print(f"    {_format_bytes(downloaded)}" + (f" / {_format_bytes(total)} ({pct})" if total else ""),
                              end="\r", file=sys.stderr)
                        last_print = time.time()
            tmp_path.rename(dest_path)
            print(f"\n  done: {dest_path.name} ({_format_bytes(dest_path.stat().st_size)})")
            return
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ConnectionError) as exc:
            print(f"\n  attempt {attempt} failed: {exc}")
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt == retries:
                raise
            time.sleep(3 * attempt)


def verify_md5(path, expected_md5):
    """Stream-hash path and compare against expected_md5. Caches a good result next to the file
    (a .md5ok marker) so re-running this script doesn't re-hash a ~12GB file every time - if you
    ever suspect the downloaded file got corrupted after the fact, delete the .md5ok marker (or the
    zip itself) to force a real re-check.
    """
    marker = path.with_suffix(path.suffix + ".md5ok")
    if marker.exists():
        print(f"  {path.name}: MD5 previously verified (delete {marker.name} to re-check)")
        return True

    print(f"  verifying MD5 of {path.name} ({_format_bytes(path.stat().st_size)}, this can take a minute)...")
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    actual = h.hexdigest()
    if actual != expected_md5:
        print(f"  MD5 MISMATCH: expected {expected_md5}, got {actual}")
        print(f"  {path} is corrupt or incomplete - delete it and re-run this script to re-download.")
        return False
    marker.write_text("ok\n")
    print(f"  MD5 verified: {actual}")
    return True


def extract_one(zip_path, extract_root):
    """Extract zip_path into extract_root, skipping if already extracted (marker-based, not by
    checking for a specific internal folder name - see reorganize_countries() for why)."""
    marker = extract_root / ".extracted"
    if marker.exists():
        print(f"  already extracted into {extract_root}, skipping")
        return
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"  extracting {zip_path.name} -> {extract_root} (large archive, this can take several minutes)")
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"{zip_path} is corrupt (bad member: {bad}) - delete it and re-run to re-download")
        zf.extractall(extract_root)
    marker.write_text("ok\n")


def reorganize_countries(extract_root, raw_dir, wanted_countries):
    """Find each wanted country's directory somewhere inside the extracted tree and link it to
    raw_dir/<country>, so convert_voc_to_yolo.py (which expects data/raw/<country>/ folders) keeps
    working unchanged regardless of whatever top-level wrapper folder(s) FigShare's zip actually
    uses internally.

    Uses a symlink rather than copying, since the extracted tree is already ~12GB+ and duplicating
    it serves no purpose. For each country, searches for directories whose name matches one of its
    known aliases (case-insensitive) and picks the SHALLOWEST match; if more than one directory at
    that same shallowest depth matches (e.g. the zip ships both a train/ and test/ split each with
    their own per-country subfolder), this prints every candidate found and picks the first
    alphabetically - flagged clearly so you can sanity-check it, rather than silently guessing.
    This is exactly the "raw data doesn't match documentation" messiness this project is meant to
    surface, so warn rather than hide it.
    """
    raw_dir.mkdir(parents=True, exist_ok=True)
    found_any = False

    for country in wanted_countries:
        link_path = raw_dir / country
        if link_path.exists() or link_path.is_symlink():
            print(f"  {country}: {link_path} already exists, leaving as-is")
            found_any = True
            continue

        aliases_lower = {a.lower() for a in COUNTRY_ALIASES[country]}
        candidates = [
            p for p in extract_root.rglob("*")
            if p.is_dir() and p.name.lower() in aliases_lower
        ]
        if not candidates:
            print(f"  WARNING: no directory matching {COUNTRY_ALIASES[country]} found under {extract_root} "
                  f"- {country} will be missing from data/raw/. Check the extracted tree by hand "
                  f"(e.g. `find {extract_root} -iname '*{country.split('_')[0]}*' -type d`) and symlink "
                  f"it manually if this script guessed wrong.")
            continue

        min_depth = min(len(p.relative_to(extract_root).parts) for p in candidates)
        shallowest = sorted(p for p in candidates if len(p.relative_to(extract_root).parts) == min_depth)
        if len(shallowest) > 1:
            print(f"  NOTE: multiple equally-shallow matches for {country}, picking the first:")
            for p in shallowest:
                print(f"    - {p.relative_to(extract_root)}")
        chosen = shallowest[0]
        link_path.symlink_to(chosen, target_is_directory=True)
        print(f"  {country}: linked data/raw/{country} -> {chosen.relative_to(extract_root)}")
        found_any = True

    if not found_any:
        print("  WARNING: none of the requested countries were found - inspect the extracted tree "
              f"under {extract_root} directly; the assumed alias names in COUNTRY_ALIASES may not "
              "match this zip's actual layout.")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR),
                         help="root data directory (default: %(default)s)")
    parser.add_argument("--countries", nargs="+", choices=list(COUNTRY_ALIASES) + ["China"], default=None,
                         help="subset of countries to link into data/raw/ after extraction (default: all). "
                              "Does NOT reduce download size - FigShare ships one combined zip for all "
                              "six countries. Pass 'China' for both China_MotorBike and China_Drone.")
    parser.add_argument("--skip-extract", action="store_true", help="download (+ verify) only, don't unzip")
    parser.add_argument("--connections", type=int, default=16,
                         help="parallel connections for aria2c downloads (default: %(default)s, ignored "
                              "if aria2c isn't installed)")
    args = parser.parse_args(argv)

    countries = args.countries
    if countries is None:
        countries = list(COUNTRY_ALIASES)
    elif "China" in countries:
        countries = [c for c in countries if c != "China"] + ["China_MotorBike", "China_Drone"]

    data_dir = Path(args.data_dir)
    zips_dir = data_dir / "zips"
    extract_root = data_dir / "raw" / "_extracted_all"
    raw_dir = data_dir / "raw"

    zip_path = zips_dir / FIGSHARE_ZIP_NAME
    print(f"Downloading combined RDD2022 archive (~12.35GB) from FigShare into {zip_path}")
    download_one(FIGSHARE_URL, zip_path, connections=args.connections)

    if not verify_md5(zip_path, FIGSHARE_MD5):
        print("\nAborting - downloaded file failed MD5 verification. Delete the zip and re-run.")
        return 1

    if args.skip_extract:
        print("\n--skip-extract set, stopping after download+verify.")
        return 0

    print(f"\nExtracting into {extract_root}")
    extract_one(zip_path, extract_root)

    print(f"\nLinking {len(countries)} countries into {raw_dir}")
    reorganize_countries(extract_root, raw_dir, countries)

    print("\nDone. Raw data is under:", raw_dir)
    print("Next: python scripts/convert_voc_to_yolo.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/convert_voc_to_yolo.py
"""Convert RDD2022's PASCAL-VOC-XML annotations into YOLO format, and build a unified manifest
across all six countries (roadmap: infra-defect-detection, phase 1).

Why this exists rather than just pointing Ultralytics at the raw VOC XML: (1) YOLO training wants
one .txt label per image with normalized [class cx cy w h] rows, not VOC's per-object XML; (2) more
importantly for this project, we need a single manifest that records which COUNTRY every image
came from, because phase 3's whole point is training on a subset of countries and evaluating
cross-country generalization - that experiment is impossible without country labels surviving the
conversion step.

Design choice - discover files by globbing + matching by filename stem, not by assuming a fixed
"images/" + "annotations/xmls/" subfolder layout: RDD2022's own directory-structure reference file
was unreachable when this project was set up (see download_rdd2022.py's docstring), so hardcoding
an assumed layout would risk silently processing zero files if the real layout differs. Globbing
recursively is slower but correct regardless of how each country's zip is actually organized
internally.

Damage classes (from the RDD2022/CRDDC'2022 label map):
    D00 - longitudinal crack
    D10 - transverse crack
    D20 - alligator crack
    D40 - pothole
Any other class name encountered (older RDD releases had more, e.g. D01/D11/D43/D44/D50) is logged
and SKIPPED, not silently merged into the nearest class - if you see a nontrivial skip count for a
country, that's worth a manual look before assuming the data converted cleanly.

Usage:
    python scripts/convert_voc_to_yolo.py                  # converts every country under data/raw/
    python scripts/convert_voc_to_yolo.py --countries Japan Czech
"""
import argparse
import csv
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    Image = None

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_MAP = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def find_files(root, suffix):
    return sorted(p for p in root.rglob(f"*{suffix}") if p.is_file())


def parse_voc_xml(xml_path):
    """Return (image_filename, width, height, [(class_name, xmin, ymin, xmax, ymax), ...]).

    width/height come from the XML's <size> block when present; if absent or zero (seen in some
    messy real-world VOC exports), the caller falls back to opening the image with PIL - this is
    exactly the kind of "annotation doesn't quite match what a clean benchmark would give you"
    messiness the project is supposed to be diagnosing, so we handle it rather than crash on it.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename_el = root.find("filename")
    image_filename = filename_el.text.strip() if filename_el is not None and filename_el.text else xml_path.stem + ".jpg"

    size_el = root.find("size")
    width = height = 0
    if size_el is not None:
        w_el, h_el = size_el.find("width"), size_el.find("height")
        width = int(w_el.text) if w_el is not None and w_el.text else 0
        height = int(h_el.text) if h_el is not None and h_el.text else 0

    objects = []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        bnd = obj.find("bndbox")
        if name_el is None or bnd is None:
            continue
        name = (name_el.text or "").strip()
        try:
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
        except (AttributeError, TypeError, ValueError):
            continue
        objects.append((name, xmin, ymin, xmax, ymax))

    return image_filename, width, height, objects


def voc_box_to_yolo_line(class_idx, xmin, ymin, xmax, ymax, width, height):
    cx = (xmin + xmax) / 2 / width
    cy = (ymin + ymax) / 2 / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return f"{class_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def convert_country(country, raw_dir, processed_dir, manifest_rows, stats):
    xml_files = find_files(raw_dir, ".xml")
    jpg_files = find_files(raw_dir, ".jpg") + find_files(raw_dir, ".jpeg") + find_files(raw_dir, ".JPG")
    image_by_stem = {}
    for p in jpg_files:
        image_by_stem.setdefault(p.stem, p)

    if not xml_files:
        print(f"  WARNING: no .xml annotation files found under {raw_dir} - is the zip actually extracted here?")
        return

    out_images = processed_dir / country / "images"
    out_labels = processed_dir / country / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    n_ok = n_orphan_xml = n_bad_size = n_no_objects = n_skipped_class = 0

    for xml_path in xml_files:
        image_filename, width, height, objects = parse_voc_xml(xml_path)
        stem = Path(image_filename).stem

        image_path = image_by_stem.get(stem) or image_by_stem.get(xml_path.stem)
        if image_path is None:
            n_orphan_xml += 1
            continue

        if width <= 0 or height <= 0:
            if Image is None:
                n_bad_size += 1
                continue
            try:
                with Image.open(image_path) as im:
                    width, height = im.size
            except Exception:
                n_bad_size += 1
                continue

        yolo_lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            if name not in CLASS_MAP:
                n_skipped_class += 1
                stats["skipped_classes"][name] = stats["skipped_classes"].get(name, 0) + 1
                continue
            xmin, xmax = sorted((max(0, xmin), min(width, xmax)))
            ymin, ymax = sorted((max(0, ymin), min(height, ymax)))
            if xmax <= xmin or ymax <= ymin:
                continue
            yolo_lines.append(voc_box_to_yolo_line(CLASS_MAP[name], xmin, ymin, xmax, ymax, width, height))
            stats["class_counts"][name] = stats["class_counts"].get(name, 0) + 1

        if not yolo_lines:
            n_no_objects += 1
            continue

        dest_stem = f"{country}__{stem}"
        dest_image = out_images / f"{dest_stem}{image_path.suffix.lower()}"
        dest_label = out_labels / f"{dest_stem}.txt"
        if not dest_image.exists():
            dest_image.write_bytes(image_path.read_bytes())
        dest_label.write_text("\n".join(yolo_lines) + "\n")

        manifest_rows.append({
            "country": country,
            "image": str(dest_image.relative_to(processed_dir.parent)),
            "label": str(dest_label.relative_to(processed_dir.parent)),
            "width": width,
            "height": height,
            "num_objects": len(yolo_lines),
            "classes": ";".join(sorted({l.split()[0] for l in yolo_lines})),
        })
        n_ok += 1

    print(f"  {country}: {n_ok} converted, {n_orphan_xml} orphan xml (no matching image), "
          f"{n_bad_size} bad/unreadable size, {n_no_objects} had zero valid objects after class "
          f"filtering, {n_skipped_class} individual boxes skipped for unrecognized class names")
    stats["per_country"][country] = {
        "converted": n_ok, "orphan_xml": n_orphan_xml, "bad_size": n_bad_size,
        "no_objects": n_no_objects, "skipped_class_boxes": n_skipped_class,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--countries", nargs="+", default=None,
                         help="subset of country folder names under data/raw/ to convert (default: all found)")
    args = parser.parse_args(argv)

    if Image is None:
        print("NOTE: Pillow not installed - images with missing/zero <size> in their XML will be "
              "skipped instead of measured. `pip install Pillow` to handle those too.", file=sys.stderr)

    data_dir = Path(args.data_dir)
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"

    if args.countries:
        countries = args.countries
    else:
        countries = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())

    if not countries:
        print(f"No country folders found under {raw_dir} - run download_rdd2022.py first.", file=sys.stderr)
        return 1

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Converting {len(countries)} countries: {countries}")
    for country in countries:
        print(f"\n[{country}]")
        convert_country(country, raw_dir / country, processed_dir, manifest_rows, stats)

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by convert_voc_to_yolo.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name, idx in CLASS_MAP.items():
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/extract_convert_per_country.py
"""Extract + convert RDD2022 ONE COUNTRY AT A TIME, deleting each country's intermediate data as
soon as it's converted (roadmap: infra-defect-detection, phase 1 - Kaggle disk-constrained variant).

WHY THIS EXISTS (2026-09-17, v2->v3->v4): the straightforward approach - download_rdd2022.py's normal
extract_one()/reorganize_countries(), which calls zf.extractall() on the WHOLE combined zip at once
- needs the 12.35GB combined zip AND all the extracted raw VOC data on disk simultaneously, which
blew through a Kaggle notebook's working-directory quota on the first attempt.

v2 tried to fix this by extracting one country's files at a time straight out of the combined zip's
member list - but that assumed the combined zip was a flat tree of images/XML per country. It
ISN'T: FigShare's combined zip (b62bd51d2ffcfaa76c60f234f0cc2bb3, the officially-published MD5) is
actually a ZIP-OF-ZIPS - exactly 7 entries, one per country, each itself a complete nested .zip
(confirmed 2026-09-17 by actually listing the real zip's namelist on Kaggle: `RDD2022/Japan.zip`,
`RDD2022/India.zip`, `RDD2022/Czech.zip`, `RDD2022/Norway.zip`, `RDD2022/United_States.zip`,
`RDD2022/China_MotorBike.zip`, `RDD2022/China_Drone.zip`). v2's matching logic looked for a path
COMPONENT exactly equal to a country alias (e.g. a directory literally named "Japan"), which never
matched because the actual component is "Japan.zip" (a file, not a directory) - so v2 silently
converted 0 images across all 7 countries and wrote an empty manifest.csv.

v3 fixed the matching (match by filename STEM instead of path component) but still copied each
country's nested zip out to a temp file ON DISK before extracting it, while the 12.35GB combined zip
stayed on disk the whole time. That's fine for the small countries, but Norway's nested zip alone is
9.9GB - so by the time v3 reached Norway, disk needed 12.35GB (combined zip, still present, only
deleted at the very end) + already-converted data from the 5 prior countries + a 9.9GB temp copy of
Norway's nested zip, all at once. That blew past Kaggle's 19.5GiB /kaggle/working quota with
`OSError: [Errno 28] No space left on device` mid-copy - not a transient hiccup, this was guaranteed
to happen as soon as processing reached the largest country, regardless of retry.

v4 (this version) fixes the actual disk-budget problem instead of just the matching bug:
  1. Read EVERY country's nested-zip bytes into RAM first (`ZipFile.read()`, not `copyfileobj` to a
     temp file) - Kaggle's RAM (~31GB, most of it free) isn't quota-limited the way /kaggle/working
     is, so holding all 7 countries' compressed bytes (summing to the same ~12.35GB as the combined
     zip) in memory costs nothing against the disk quota.
  2. Only ONCE ALL SEVEN have been read into memory - and the combined zip is no longer needed for
     anything - close it and delete the 12.35GB file from disk, BEFORE extracting a single country to
     disk. This frees the full 19.5GiB quota (minus whatever's already used) for the extraction step,
     regardless of which country happens to be biggest or what order they're processed in.
  3. Per country: extract from an in-memory `io.BytesIO` (no on-disk temp zip at all), convert, delete
     the extracted raw folder, and drop that country's bytes from the in-memory dict - so RAM usage
     shrinks back down as we go instead of holding all 7 for the whole run.

Usage:
    python scripts/extract_convert_per_country.py                  # all countries, then deletes the combined zip
    python scripts/extract_convert_per_country.py --keep-zip       # don't delete the combined zip early or at the end
                                                                    # (only use this if you have >32GB of quota - it
                                                                    # defeats the whole point of the v4 fix above)
    python scripts/extract_convert_per_country.py --data-dir data
"""
import argparse
import csv
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import download_rdd2022 as dl          # noqa: E402  (FIGSHARE_ZIP_NAME, COUNTRY_ALIASES)
import convert_voc_to_yolo as cv       # noqa: E402  (convert_country, CLASS_MAP, CLASS_NAMES)


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _print_disk_usage(label, working_dir=None):
    result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
    print(f"  [{label}] df -h /:\n" + "\n".join("    " + l for l in result.stdout.splitlines()))
    # df -h / reports the container's overall overlay filesystem, which on Kaggle does NOT move in
    # step with /kaggle/working - the thing actually counted against the 19.5GiB quota is disk usage
    # UNDER /kaggle/working specifically, which `du` reports correctly.
    if working_dir is not None:
        du = subprocess.run(["du", "-sh", str(working_dir)], capture_output=True, text=True)
        print(f"  [{label}] du -sh {working_dir}: {du.stdout.strip() or du.stderr.strip()}")


def match_country_by_stem(entry_name):
    """Which COUNTRY_ALIASES key this combined-zip entry is, by comparing its filename stem (the
    name with the LAST extension stripped, e.g. "RDD2022/China_MotorBike.zip" -> "China_MotorBike")
    against the known aliases, case-insensitively. This is the v3 fix: v2 matched whole path
    COMPONENTS looking for a directory named e.g. "Japan", which never matched because the real
    entries are files named "Japan.zip", not directories named "Japan"."""
    stem_lower = Path(entry_name).stem.lower()
    for country, aliases in dl.COUNTRY_ALIASES.items():
        for alias in aliases:
            if alias.lower() == stem_lower:
                return country
    return None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--keep-zip", action="store_true",
                         help="don't delete the combined zip early (once all countries are read into "
                              "memory) or at the end - default is to delete it early, which is what "
                              "makes the largest country (Norway, ~9.9GB) fit under Kaggle's quota")
    parser.add_argument("--countries", nargs="+", default=None,
                         help="only extract+convert these countries (default: all 7). Added for phase 2, "
                              "which only needs a single country (e.g. --countries Czech) - still reads "
                              "the whole combined zip's member list, but only pulls the requested "
                              "countries' bytes into memory, so the early-delete-the-zip step still runs "
                              "and disk usage stays tiny.")
    args = parser.parse_args(argv)

    data_dir = Path(args.data_dir)
    zip_path = data_dir / "zips" / dl.FIGSHARE_ZIP_NAME
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"
    working_dir = data_dir.resolve().parent  # e.g. /kaggle/working - what the quota actually tracks

    if not zip_path.exists():
        print(f"{zip_path} not found - run download_rdd2022.py --skip-extract first.", file=sys.stderr)
        return 1

    # Clean slate for raw_dir: it's always transient intermediate storage, never a final deliverable,
    # so wipe any leftover partial state from a previous crashed run (e.g. a half-written
    # _nested_Norway.zip from the v3 run that hit the disk-quota error) before starting.
    if raw_dir.exists():
        print(f"Removing leftover {raw_dir} from a previous run (transient data only, safe to wipe)...")
        shutil.rmtree(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Opening {zip_path} to read its member list (no extraction yet, costs no disk space)...")
    outer_zf = zipfile.ZipFile(zip_path)
    infos = [i for i in outer_zf.infolist() if not i.filename.endswith("/")]
    print(f"  combined zip contains {len(infos)} entries")

    by_country = {}
    unmatched = []
    for info in infos:
        country = match_country_by_stem(info.filename)
        if country:
            by_country[country] = info
        else:
            unmatched.append(info.filename)

    print("\nEntries matched (this is the combined zip's REAL layout - one nested zip per country):")
    for country in dl.COUNTRY_ALIASES:
        if country in by_country:
            info = by_country[country]
            print(f"  {country}: {info.filename} ({_format_bytes(info.file_size)})")
        else:
            print(f"  {country}: NOT FOUND")

    if args.countries:
        missing = [c for c in args.countries if c not in by_country]
        if missing:
            print(f"\n--countries requested {missing} but the combined zip doesn't have (a match for) "
                  f"{'them' if len(missing) > 1 else 'it'} - check spelling against COUNTRY_ALIASES.", file=sys.stderr)
            return 1
        by_country = {c: by_country[c] for c in args.countries}
        print(f"\n--countries set: only extracting {list(by_country.keys())} (out of all 7 found above)")

    if unmatched:
        print(f"\n  {len(unmatched)} entries matched no known country alias:")
        for n in unmatched:
            print(f"    {n}")

    _print_disk_usage("before reading anything", working_dir)

    # Step 1: read EVERY country's nested-zip bytes into RAM before extracting any of them to disk.
    # This is what lets us delete the 12.35GB combined zip BEFORE the largest country (Norway,
    # 9.9GB) needs to be extracted, instead of only being able to delete it at the very end.
    print(f"\nReading all {len(by_country)} countries' nested-zip bytes into memory (RAM isn't "
          f"quota-limited the way /kaggle/working is - this avoids ever needing the 12.35GB combined "
          f"zip and a country's extracted data on disk at the same time)...")
    country_bytes = {}
    for country, info in by_country.items():
        print(f"  reading {country} ({info.filename}, {_format_bytes(info.file_size)}) into memory...")
        country_bytes[country] = outer_zf.read(info)
    zip_size = zip_path.stat().st_size
    outer_zf.close()

    if not args.keep_zip:
        zip_path.unlink()
        print(f"\nDeleted {zip_path} ({_format_bytes(zip_size)}) early - every country's bytes are "
              f"already in memory, so the combined zip is no longer needed and this frees up disk "
              f"headroom before extracting any country.")
    else:
        print(f"\n--keep-zip set: leaving {zip_path} ({_format_bytes(zip_size)}) on disk (less headroom "
              f"for the extraction step below - only safe with a much larger quota than 19.5GiB).")
    _print_disk_usage("after reading all countries into memory" + ("" if args.keep_zip else " + deleting the combined zip"), working_dir)

    # Step 2: per country, extract from the in-memory bytes (no on-disk temp zip), convert, clean up.
    for country in list(country_bytes.keys()):
        data = country_bytes.pop(country)  # drop from the dict now so RAM shrinks as we go

        country_raw = raw_dir / country
        if country_raw.exists():
            shutil.rmtree(country_raw)
        country_raw.mkdir(parents=True)

        print(f"\n[{country}] extracting {_format_bytes(len(data))} (in memory) into {country_raw}...")
        with zipfile.ZipFile(io.BytesIO(data)) as inner_zf:
            bad = inner_zf.testzip()
            if bad is not None:
                print(f"  WARNING: {country}'s nested zip is corrupt (bad member: {bad}) - skipping {country}")
                shutil.rmtree(country_raw)
                del data
                continue
            inner_zf.extractall(country_raw)
        del data  # free this country's RAM now that it's on disk as extracted files

        print(f"  converting...")
        cv.convert_country(country, country_raw, processed_dir, manifest_rows, stats)

        print(f"  deleting {country_raw} (already converted, no longer needed) to free space for the next country...")
        shutil.rmtree(country_raw)
        _print_disk_usage(f"after {country}", working_dir)

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by extract_convert_per_country.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(cv.CLASS_NAMES)}\n"
        f"names: {cv.CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name in cv.CLASS_MAP:
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    if not manifest_rows:
        print("\nWARNING: manifest is EMPTY - 0 images were converted. Check the 'Entries matched' listing "
              "above for NOT FOUND countries or unmatched entries before trusting anything downstream.")

    _print_disk_usage("final", working_dir)
    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/make_country_split.py
"""Build a reproducible train/val/test split for a single country's data, from manifest.csv
(roadmap: infra-defect-detection, phase 2 - CNN baseline).

Why a separate script rather than letting Ultralytics auto-split: phase 2's whole point is to
establish a "same-country train/test" baseline number that phase 3 later compares a cross-country
number against - that comparison is only meaningful if the split is fixed, recorded, and re-usable
(so phase 3's report can say "the same Czech test set" rather than "some random subset"). Ultralytics
can auto-split a folder, but doesn't write out a reproducible, inspectable record of which images
ended up in which split - so we do it ourselves and hand Ultralytics explicit image-list .txt files.

Usage:
    python scripts/make_country_split.py --country Czech
    python scripts/make_country_split.py --country Czech --train-frac 0.7 --val-frac 0.15 --seed 42
"""
import argparse
import csv
import json
import random
import sys
from pathlib import Path

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--country", required=True, help="country name as it appears in manifest.csv's country column")
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--train-frac", type=float, default=0.7)
    parser.add_argument("--val-frac", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42, help="fixed shuffle seed - record this alongside any reported metric")
    args = parser.parse_args(argv)

    if args.train_frac + args.val_frac >= 1.0:
        print(f"--train-frac ({args.train_frac}) + --val-frac ({args.val_frac}) must leave room for a "
              f"non-empty test split (currently sums to {args.train_frac + args.val_frac})", file=sys.stderr)
        return 1

    data_dir = Path(args.data_dir)
    manifest_path = data_dir / "manifest.csv"
    if not manifest_path.exists():
        print(f"{manifest_path} not found - run extract_convert_per_country.py first.", file=sys.stderr)
        return 1

    with open(manifest_path, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["country"] == args.country]

    if not rows:
        print(f"No rows for country={args.country!r} in {manifest_path} - check spelling, or that "
              f"extract_convert_per_country.py was run with --countries {args.country}.", file=sys.stderr)
        return 1

    # Sort first so the shuffle is deterministic regardless of manifest.csv's row order (which
    # itself depends on filesystem iteration order and isn't guaranteed stable across re-runs).
    rows.sort(key=lambda r: r["image"])
    rng = random.Random(args.seed)
    rng.shuffle(rows)

    n = len(rows)
    n_train = int(n * args.train_frac)
    n_val = int(n * args.val_frac)
    splits = {
        "train": rows[:n_train],
        "val": rows[n_train:n_train + n_val],
        "test": rows[n_train + n_val:],
    }

    split_dir = data_dir / "splits" / args.country
    split_dir.mkdir(parents=True, exist_ok=True)

    for split_name, split_rows in splits.items():
        list_path = split_dir / f"{split_name}.txt"
        with open(list_path, "w") as f:
            for r in split_rows:
                # manifest.csv's "image" column is already relative to data_dir (e.g.
                # "processed/Czech/images/Czech__xxx.jpg") - Ultralytics wants absolute paths to be
                # safe regardless of what directory `yolo train` is invoked from.
                f.write(str((data_dir / r["image"]).resolve()) + "\n")
        print(f"  {split_name}: {len(split_rows)} images -> {list_path}")

    if any(len(v) == 0 for v in splits.values()):
        print(f"\nWARNING: at least one split is empty (train={len(splits['train'])}, "
              f"val={len(splits['val'])}, test={len(splits['test'])}) - {args.country} may not have "
              f"enough images for these fractions.", file=sys.stderr)

    dataset_yaml_path = split_dir / "dataset.yaml"
    dataset_yaml_path.write_text(
        f"# Auto-generated by make_country_split.py for country={args.country}, seed={args.seed}.\n"
        f"# Labels are found by Ultralytics' convention: same path with the LAST 'images' path "
        f"component replaced by 'labels' and the extension replaced by .txt - this matches how "
        f"convert_voc_to_yolo.py laid out data/processed/{args.country}/images/ + labels/.\n"
        f"train: {(split_dir / 'train.txt').resolve()}\n"
        f"val: {(split_dir / 'val.txt').resolve()}\n"
        f"test: {(split_dir / 'test.txt').resolve()}\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )
    print(f"  dataset.yaml -> {dataset_yaml_path}")

    split_config = {
        "country": args.country,
        "seed": args.seed,
        "train_frac": args.train_frac,
        "val_frac": args.val_frac,
        "test_frac": round(1.0 - args.train_frac - args.val_frac, 6),
        "n_total": n,
        "n_train": len(splits["train"]),
        "n_val": len(splits["val"]),
        "n_test": len(splits["test"]),
    }
    config_path = split_dir / "split_config.json"
    config_path.write_text(json.dumps(split_config, indent=2) + "\n")
    print(f"  split_config.json -> {config_path}")
    print(f"\n{args.country}: {n} total images -> train={len(splits['train'])} "
          f"val={len(splits['val'])} test={len(splits['test'])} (seed={args.seed})")
    print(f"\nNext: python scripts/train_cnn_baseline.py --data {dataset_yaml_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/train_cnn_baseline.py
"""Train + evaluate a single-country YOLO11 baseline (roadmap: infra-defect-detection, phase 2).

WHY THIS EXISTS: phase 2's job is to establish a "same-country train/test" detection baseline
BEFORE phase 3 asks "how much worse does this get when train and test countries differ" - that
comparison is only meaningful if this baseline is: (1) evaluated on a held-out TEST split the model
never saw during training or checkpoint selection (not the 'val' split Ultralytics uses internally
to pick the best epoch - that would leak), (2) reported as more than one aggregate number (mAP alone
hides whether the model just gives up on one class entirely), and (3) fully reproducible (seed, split
fractions, hyperparameters all recorded next to the numbers, not just in this script's defaults).

Usage:
    python scripts/train_cnn_baseline.py --data data/splits/Czech/dataset.yaml
    python scripts/train_cnn_baseline.py --data data/splits/Czech/dataset.yaml --model yolo11s.pt --epochs 100
"""
import argparse
import json
import shutil
import sys
import time
from pathlib import Path

import yaml
from ultralytics import YOLO

DEFAULT_RUNS_DIR = Path(__file__).resolve().parent.parent / "runs" / "phase2"


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", required=True, help="path to a dataset.yaml written by make_country_split.py")
    parser.add_argument("--model", default="yolo11n.pt",
                         help="Ultralytics model/checkpoint to start from (default: yolo11n.pt, the "
                              "smallest pretrained YOLO11 - appropriate for a baseline on a small "
                              "single-country dataset and Kaggle's free T4 quota). Use a bare "
                              "'yolo11n.yaml' instead to train from random init with no internet "
                              "download, if the pretrained-weights download is ever unavailable.")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--seed", type=int, default=42, help="must match (or be recorded alongside) the seed used for the split")
    parser.add_argument("--patience", type=int, default=20, help="early-stop patience (epochs with no val improvement)")
    parser.add_argument("--project", default=str(DEFAULT_RUNS_DIR))
    parser.add_argument("--name", default=None, help="run name (default: derived from the country in --data)")
    args = parser.parse_args(argv)

    data_yaml_path = Path(args.data)
    if not data_yaml_path.exists():
        print(f"{data_yaml_path} not found - run make_country_split.py first.", file=sys.stderr)
        return 1

    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    class_names = data_cfg.get("names", [])

    # Pull the split's own recorded config (seed/fractions/counts) if make_country_split.py wrote one
    # alongside dataset.yaml, so the final report shows the ACTUAL split used, not just this script's
    # own --seed default (which only controls training, not how images were assigned to splits).
    split_config_path = data_yaml_path.parent / "split_config.json"
    split_config = json.loads(split_config_path.read_text()) if split_config_path.exists() else None
    country = (split_config or {}).get("country") or data_yaml_path.parent.name
    run_name = args.name or f"{country.lower()}_baseline"

    print(f"Training {args.model} on {country} (data={data_yaml_path}, seed={args.seed}, "
          f"epochs={args.epochs}, imgsz={args.imgsz})...")
    if split_config:
        print(f"  split (from {split_config_path.name}): train={split_config['n_train']} "
              f"val={split_config['n_val']} test={split_config['n_test']} "
              f"(split seed={split_config['seed']}, fractions "
              f"{split_config['train_frac']}/{split_config['val_frac']}/{split_config['test_frac']})")

    model = YOLO(args.model)
    t0 = time.time()
    model.train(
        data=str(data_yaml_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        seed=args.seed,
        patience=args.patience,
        project=args.project,
        name=run_name,
        exist_ok=True,
        plots=True,
    )
    train_seconds = time.time() - t0

    # Evaluate on the held-out TEST split explicitly - Ultralytics' training loop only ever looks at
    # 'val' (for early stopping / best-checkpoint selection), so this is the model's first and only
    # exposure to test images, which is what makes this number usable as phase 3's baseline to beat.
    print(f"\nEvaluating best checkpoint on the TEST split (never seen during training)...")
    test_results = model.val(data=str(data_yaml_path), split="test", plots=True)
    run_dir = Path(test_results.save_dir)

    box = test_results.box
    per_class = []
    for idx in range(len(class_names)):
        if idx in box.ap_class_index:
            p, r, ap50, ap = box.class_result(list(box.ap_class_index).index(idx))
            per_class.append({"class": class_names[idx], "precision": float(p), "recall": float(r),
                               "ap50": float(ap50), "ap50_95": float(ap)})
        else:
            per_class.append({"class": class_names[idx], "precision": None, "recall": None,
                               "ap50": None, "ap50_95": None, "note": "no test-set instances of this class"})

    metrics = {
        "country": country,
        "model": args.model,
        "seed": args.seed,
        "epochs": args.epochs,
        "imgsz": args.imgsz,
        "batch": args.batch,
        "train_seconds": round(train_seconds, 1),
        "split_config": split_config,
        "overall": {
            "precision_mean": float(box.mp),
            "recall_mean": float(box.mr),
            "map50": float(box.map50),
            "map50_95": float(box.map),
            "map75": float(box.map75),
        },
        "per_class": per_class,
        "train_run_dir": str(Path(args.project) / run_name),
        "val_run_dir": str(run_dir),
    }

    metrics_path = run_dir / "baseline_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

    report_lines = [
        f"# Phase 2 CNN baseline - {country}",
        "",
        f"Generated by `scripts/train_cnn_baseline.py`. Same-country train/test baseline - phase 3's "
        f"cross-country experiment reports how much this degrades when train and test countries differ.",
        "",
        "## Config (for reproducibility)",
        "",
        f"- model: `{args.model}`",
        f"- seed: {args.seed}",
        f"- epochs: {args.epochs} (patience={args.patience})",
        f"- imgsz: {args.imgsz}, batch: {args.batch}",
        f"- train time: {train_seconds / 60:.1f} min",
    ]
    if split_config:
        report_lines += [
            f"- split: train={split_config['n_train']} val={split_config['n_val']} "
            f"test={split_config['n_test']} out of {split_config['n_total']} total "
            f"(seed={split_config['seed']}, fractions "
            f"{split_config['train_frac']}/{split_config['val_frac']}/{split_config['test_frac']})",
        ]
    report_lines += [
        "",
        "## Overall metrics (on the held-out TEST split)",
        "",
        "| metric | value |",
        "|---|---|",
        f"| mAP@50 | {box.map50:.4f} |",
        f"| mAP@50-95 | {box.map:.4f} |",
        f"| mAP@75 | {box.map75:.4f} |",
        f"| mean precision | {box.mp:.4f} |",
        f"| mean recall | {box.mr:.4f} |",
        "",
        "## Per-class precision / recall / AP (on the held-out TEST split)",
        "",
        "| class | precision | recall | AP50 | AP50-95 |",
        "|---|---|---|---|---|",
    ]
    for c in per_class:
        if c["precision"] is None:
            report_lines.append(f"| {c['class']} | - | - | - | - ({c.get('note', '')}) |")
        else:
            report_lines.append(f"| {c['class']} | {c['precision']:.4f} | {c['recall']:.4f} | "
                                 f"{c['ap50']:.4f} | {c['ap50_95']:.4f} |")
    train_run_dir = Path(args.project) / run_name
    report_lines += [
        "",
        "## Artifacts",
        "",
        f"- confusion matrix: `{run_dir / 'confusion_matrix.png'}` (raw counts) and "
        f"`{run_dir / 'confusion_matrix_normalized.png'}`",
        f"- PR / F1 / precision / recall curves: `{run_dir}/Box{{PR,F1,P,R}}_curve.png`",
        f"- training curves (loss/mAP per epoch): `{train_run_dir / 'results.png'}` "
        f"(from the `train` run at `{train_run_dir}`, not this `val` run directory)",
        f"- machine-readable metrics: `{metrics_path}`",
        "",
    ]
    report_path = run_dir / "baseline_report.md"
    report_path.write_text("\n".join(report_lines) + "\n")

    print(f"\nWrote {metrics_path}")
    print(f"Wrote {report_path}")
    print(f"\nOverall: mAP50={box.map50:.4f} mAP50-95={box.map:.4f} "
          f"(precision={box.mp:.4f}, recall={box.mr:.4f})")
    if any(c["precision"] is None for c in per_class):
        missing = [c["class"] for c in per_class if c["precision"] is None]
        print(f"\nNOTE: {missing} had no instances in the test split - their AP is undefined, not "
              f"zero. Look at split_config.json / re-run make_country_split.py with a different seed "
              f"if this class is important and the test set is just too small to contain it by chance.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 1. 安装依赖

In [ ]:
!pip install -q ultralytics pandas Pillow pyyaml
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable the GPU accelerator in Session options!)")


## 2. 下载 + MD5校验

FigShare只提供一个12.35GB的合并压缩包(zip-of-zips,7个国家各一个内嵌zip),没法只下载Czech那一份,所以这一步还是要下完整个包。如果之前跑过phase 1、`data/zips/`里已经有校验过的zip,这一步会直接跳过下载。

In [ ]:
!python scripts/download_rdd2022.py --skip-extract

## 3. 按国家分批解压转换,只跑Czech

只把Czech的nested zip读进内存、解压、转换,合并zip读完所有请求的国家后立刻删除(这里只请求Czech一个,所以立刻就会删,不会占满磁盘)。

In [ ]:
!python scripts/extract_convert_per_country.py --countries Czech

## 4. 划分train/val/test

固定随机种子(默认42),70/15/15划分,写入`data/splits/Czech/`下的train.txt/val.txt/test.txt + dataset.yaml + split_config.json(记录划分参数,供报告和phase 3复现引用)。

In [ ]:
!python scripts/make_country_split.py --country Czech --seed 42

## 5. 训练 + 评测

用YOLO11n(最小的预训练权重,适合这个数据量级和免费GPU额度)训练,epochs=100但patience=20(20轮没提升就早停,不用死等100轮)。训练完在**从没见过的test集**上跑一次`model.val()`,这是baseline要报的真实数字,不是训练过程中Ultralytics自己用来挑最优checkpoint的val集。

In [ ]:
!python scripts/train_cnn_baseline.py \
    --data data/splits/Czech/dataset.yaml \
    --model yolo11n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --seed 42


## 6. 核对结果

In [ ]:
!echo '--- split_config.json ---'
!cat data/splits/Czech/split_config.json
!echo
!echo '--- baseline_report.md ---'
!find runs -name baseline_report.md -exec cat {} \;
!echo
!echo '--- confusion matrix / curves exist? ---'
!find runs -name "confusion_matrix*.png" -o -name "results.png"


## 7. 收尾:提交并把小文件带回Mac

需要带回Mac的文件(都很小):
- `data/splits/Czech/`(train.txt/val.txt/test.txt/dataset.yaml/split_config.json)
- `runs/phase2/czech_baseline/results.png`、`weights/best.pt`(模型权重,几MB,YOLO11n很小)
- `runs/detect/val/baseline_report.md`、`baseline_metrics.json`、`confusion_matrix.png`、`confusion_matrix_normalized.png`

在右侧Output文件树里找到这些文件,用kebab菜单(⋮)逐个Download,和phase 1收尾时一样。